# 03c Embeddings v3 No Log Target

This secondary notebook builds the no-log embedding datasets. The target attached to each pooled dataset is `systemic_risk_label`, not `log_systemic_risk_label`.

The embedding models themselves are still trained as unsupervised / self-supervised representation models. The no-log change affects the downstream target stored with the embeddings.

In [ ]:
import sys
import os
from pathlib import Path

%matplotlib inline

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from src.models.fix_embeddings import FixedGNNConfig, build_fixed_pooled_dataset
from src.models.embeddings import Node2VecConfig

PROJECT_ROOT = Path().resolve().parents[1]
TARGET_DIR = PROJECT_ROOT / "src" / "datasets" / "targets"
OUTPUT_DIR = PROJECT_ROOT / "src" / "data" / "embeddings"

pd.set_option("display.max_columns", 200)
print(f"Project root: {PROJECT_ROOT}")
print(f"Target dir:   {TARGET_DIR}")

## Configuration

Four datasets are generated:

- GraphSAGE reconstruction embeddings, 32 dimensions
- GraphSAGE reconstruction embeddings, 64 dimensions
- Node2Vec structural embeddings, 32 dimensions
- Node2Vec structural embeddings, 64 dimensions

In [ ]:
TARGET_COL = "systemic_risk_label"
INCLUDE_RAW_FEATURES = False
YEARS = range(2016, 2024)
QUARTERS = (1, 2, 3, 4)

cfg_graphsage_32 = FixedGNNConfig(
    hidden_dims=(256, 32),
    dropout=0.3,
    lr=0.01,
    epochs=100,
    reconstruction_weight=1.0,
    link_weight=0.0,
    aggregation="mean",
    device="cpu",
)

cfg_graphsage_64 = FixedGNNConfig(
    hidden_dims=(256, 64),
    dropout=0.3,
    lr=0.01,
    epochs=100,
    reconstruction_weight=1.0,
    link_weight=0.0,
    aggregation="mean",
    device="cpu",
)

cfg_node2vec_32 = Node2VecConfig(
    embedding_dim=32,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)

cfg_node2vec_64 = Node2VecConfig(
    embedding_dim=64,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)

OUTPUTS = {
    "graphsage_32": OUTPUT_DIR / "graphsage_fixed_32_srisk_nolog_dataset.parquet",
    "graphsage_64": OUTPUT_DIR / "graphsage_fixed_64_srisk_nolog_dataset.parquet",
    "node2vec_32": OUTPUT_DIR / "node2vec_fixed_32_srisk_nolog_dataset.parquet",
    "node2vec_64": OUTPUT_DIR / "node2vec_fixed_64_srisk_nolog_dataset.parquet",
}

OUTPUTS

## Build GraphSAGE 32-Dim No-Log Dataset

In [ ]:
pooled_graphsage_32 = build_fixed_pooled_dataset(
    config=cfg_graphsage_32,
    years=YEARS,
    quarters=QUARTERS,
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    target_dir=TARGET_DIR,
    output_path=OUTPUTS["graphsage_32"],
)

pooled_graphsage_32.shape

In [ ]:
embedding_cols = [c for c in pooled_graphsage_32.columns if c.startswith("emb_")]
print(f"Embedding columns: {len(embedding_cols)}")
pooled_graphsage_32.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

## Build GraphSAGE 64-Dim No-Log Dataset

In [ ]:
pooled_graphsage_64 = build_fixed_pooled_dataset(
    config=cfg_graphsage_64,
    years=YEARS,
    quarters=QUARTERS,
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    target_dir=TARGET_DIR,
    output_path=OUTPUTS["graphsage_64"],
)

pooled_graphsage_64.shape

In [ ]:
embedding_cols = [c for c in pooled_graphsage_64.columns if c.startswith("emb_")]
print(f"Embedding columns: {len(embedding_cols)}")
pooled_graphsage_64.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

## Build Node2Vec 32-Dim No-Log Dataset

In [ ]:
pooled_node2vec_32 = build_fixed_pooled_dataset(
    config=cfg_node2vec_32,
    years=YEARS,
    quarters=QUARTERS,
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    target_dir=TARGET_DIR,
    output_path=OUTPUTS["node2vec_32"],
)

pooled_node2vec_32.shape

In [ ]:
embedding_cols = [c for c in pooled_node2vec_32.columns if c.startswith("emb_")]
print(f"Embedding columns: {len(embedding_cols)}")
pooled_node2vec_32.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

## Build Node2Vec 64-Dim No-Log Dataset

In [ ]:
pooled_node2vec_64 = build_fixed_pooled_dataset(
    config=cfg_node2vec_64,
    years=YEARS,
    quarters=QUARTERS,
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    target_dir=TARGET_DIR,
    output_path=OUTPUTS["node2vec_64"],
)

pooled_node2vec_64.shape

In [ ]:
embedding_cols = [c for c in pooled_node2vec_64.columns if c.startswith("emb_")]
print(f"Embedding columns: {len(embedding_cols)}")
pooled_node2vec_64.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

## Saved Files

In [ ]:
for name, path in OUTPUTS.items():
    print(f"{name:14s} -> {path}")